In [ ]:
from collections import defaultdict
import itertools
from nltk import ngrams
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import sent_tokenize
import numpy as np
import pandas as pd
import pickle
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
from typing import List, Union, Optional

from dap_job_quality import config, PROJECT_DIR, logging
from dap_job_quality.getters.afs_data import get_eyp_ads, get_sim_occ_ads

# Load BERT model and tokenizer
model_name = config["sentence_model"]

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [ ]:
# Mean Pooling - Take attention mask into account for correct averaging
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[
        0
    ]  # First element of model_output contains all token embeddings
    input_mask_expanded = (
        attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    )
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    return sum_embeddings / sum_mask


def embed_sentences(
    sentences: List[str], model_name: str = model_name, batch_size: int = 32
) -> List[torch.Tensor]:
    """
    Generate embeddings for each sentence in a list of sentences using a specified model.

    Follows the method described here: https://www.sbert.net/examples/applications/computing-embeddings/README.html

    Args:
        sentences (List[str]): A list of sentences to be embedded.
        model_name (str): The name of the model to use for generating embeddings. Default
                          is a globally defined variable `SENT_MODEL`.

    Returns:
        List[torch.Tensor]: A list of tensors where each tensor represents the embedding
                            of a corresponding sentence in the input list.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)

    embeddings = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i + batch_size]
        encoded_input = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors="pt")
        
        # Compute token embeddings
        with torch.no_grad():
            model_output = model(**encoded_input)

        # Perform pooling. In this case, mean pooling
        batch_embeddings = mean_pooling(model_output, encoded_input["attention_mask"])
        embeddings.append(batch_embeddings)

    return torch.cat(embeddings, dim=0)

def split_ngrams(text, length=6, n=4):
    if len(text.split()) > length:
        ngram_list = list(ngrams(text.split(), n))
    else:
        ngram_list = [text]
    return ngram_list

In [ ]:
eyp = get_eyp_ads()#[['id', 'clean_description']]
sim_occs = get_sim_occ_ads()#[['id', 'clean_description']]
all_job_ads = pd.concat([eyp, sim_occs], axis=0).drop_duplicates()

lookup = pd.read_csv(PROJECT_DIR / "inputs/keyword_lookup - v5.csv")

# Filter sentences which relate to job quality

In [ ]:
logging.info(len(all_job_ads))
all_job_ads = all_job_ads[all_job_ads['created']>='2022-01-01']
logging.info(len(all_job_ads))
job_ads_sample = all_job_ads.sample(1000, random_state=42)

In [ ]:
job_ads_sample['sentences'] = job_ads_sample['clean_description'].apply(sent_tokenize)
job_ads_sample = job_ads_sample.explode('sentences')
logging.info(f'{len(job_ads_sample)} sentences')

In [ ]:
with open(PROJECT_DIR / 'outputs/models/sentence_classifier/logistic_regression.pkl', 'rb') as file:
    model = pickle.load(file)

# Load the saved PCA transformer (if used)
with open(PROJECT_DIR / 'outputs/models/sentence_classifier/pca.pkl', 'rb') as file:
    pca = pickle.load(file)

In [ ]:
ad_embeddings = embed_sentences(job_ads_sample['sentences'].tolist())

In [ ]:
X_new_pca = pca.transform(ad_embeddings)

predictions = model.predict(X_new_pca)

# Get ngrams from JQ sentences

In [ ]:
all_job_ads['ngrams'] = all_job_ads['sentences'].apply(lambda x: split_ngrams(x, 6, 4))
sentence_df_long = all_job_ads.explode('ngrams')
sentence_df_long['ngrams'] = sentence_df_long['ngrams'].apply(lambda x: ' '.join(x) if isinstance(x, tuple) else x)
sentence_df_long.head()

# Calculate cosine similarity of unique ngrams to target phrases

In [ ]:
unique_ngrams = list(sentence_df_long['ngrams'].unique())

In [ ]:
# Embed the target phrases
target_embeddings = embed_sentences(target_phrases)
    
ngram_embeddings = embed_sentences(unique_ngrams)
        
similarities = calculate_cosine_similarity(ngram_embeddings, target_embeddings)

similarities
        
matches = defaultdict(list)
for i, ngram in enumerate(unique_ngrams):
    for j, target_phrase in enumerate(target_phrases):
        if similarities[i, j] > 0.8:
            matches[ngram].append((target_phrase, similarities[i, j]))
            
# Deduplicate target phrases for each text
deduplicated_matches = {}
for ngram, matches_list in matches.items():
    unique_matches = list({phrase: sim for phrase, sim in matches_list}.items())
    deduplicated_matches[ngram] = unique_matches

In [ ]:
matches_df = pd.DataFrame(deduplicated_matches.items(), columns=['ngram', 'matches'])

In [ ]:
sentence_df_long = pd.merge(sentence_df_long, matches_df, how='left', left_on='ngrams', right_on='ngram')
sentence_df_long.head()